# 03 — Feature Engineering + Random Forest Training

**Purpose:** Build training dataset, train Random Forest untuk predict onset DOY, validate dengan Leave-One-Out CV, save model.

## Features
1. `oni_jjas_avg` — average ONI Juni-September (dry season ENSO state)
2. `oni_lag_3` — ONI 3 bulan sebelum typical onset (~Juli)
3. `pre_season_rainfall` — total rainfall Juni-Juli
4. `prev_year_onset_doy` — onset DOY tahun sebelumnya

## Target
- `onset_doy` — day of year of detected onset

## Validation
- Leave-One-Out Cross-Validation (n=30, ideal for small dataset)
- Reported metric: MAE (Mean Absolute Error in days)

## Outputs
- `data/processed/training_features.csv` — training dataset
- `data/processed/loo_cv_results.csv` — LOO predictions vs actuals
- `model/onset_predictor.pkl` — trained RandomForest
- `model/feature_columns.json` — feature column names (for inference)
- `model/model_metadata.json` — MAE, training date, etc.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LinearRegression
from datetime import datetime
import joblib
import warnings
warnings.filterwarnings('ignore')

DATA_PROCESSED = Path('../data/processed')
MODEL_DIR = Path('../model')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load all preprocessed data
onset_df = pd.read_csv(DATA_PROCESSED / 'onset_dates.csv')
onset_df['onset_date'] = pd.to_datetime(onset_df['onset_date'])

oni_df = pd.read_csv(DATA_PROCESSED / 'oni_monthly.csv')

rainfall_df = pd.read_csv(DATA_PROCESSED / 'rainfall_daily_clean.csv')
rainfall_df['date'] = pd.to_datetime(rainfall_df['date'])

print(f"Onset events: {len(onset_df)}")
print(f"ONI records: {len(oni_df)}")
print(f"Rainfall days: {len(rainfall_df)}")

## Feature Engineering

In [ ]:
def build_features(year, onset_df, oni_df, rainfall_df):
    """Build feature row for given year."""
    
    # 1. ONI JJAS average (Juni-September)
    jjas = oni_df[(oni_df['year'] == year) & (oni_df['month'].isin([6, 7, 8, 9]))]
    oni_jjas_avg = jjas['oni_value'].mean() if len(jjas) > 0 else np.nan
    
    # 2. ONI lag-3 (ONI bulan Juli, 3 bulan sebelum typical onset Oktober)
    july_oni = oni_df[(oni_df['year'] == year) & (oni_df['month'] == 7)]
    oni_lag_3 = july_oni['oni_value'].values[0] if len(july_oni) > 0 else np.nan
    
    # 3. Pre-season rainfall (Juni-Juli total)
    pre_season = rainfall_df[
        (rainfall_df['date'].dt.year == year) &
        (rainfall_df['date'].dt.month.isin([6, 7]))
    ]
    pre_season_rainfall = pre_season['rainfall_mm'].sum()
    
    # 4. Previous year onset DOY
    prev = onset_df[onset_df['year'] == year - 1]
    prev_year_onset_doy = prev['onset_doy'].values[0] if len(prev) > 0 else 295  # default to traditional
    
    return {
        'year': year,
        'oni_jjas_avg': oni_jjas_avg,
        'oni_lag_3': oni_lag_3,
        'pre_season_rainfall': pre_season_rainfall,
        'prev_year_onset_doy': prev_year_onset_doy
    }

# Build feature dataset for all detected onset years
features_list = []
for _, row in onset_df.iterrows():
    feats = build_features(row['year'], onset_df, oni_df, rainfall_df)
    feats['onset_doy'] = row['onset_doy']
    features_list.append(feats)

features_df = pd.DataFrame(features_list)
features_df = features_df.dropna()  # drop years with missing features
print(f"Feature dataset shape: {features_df.shape}")
features_df.head()

In [ ]:
# Save features
features_df.to_csv(DATA_PROCESSED / 'training_features.csv', index=False)
print(f"✅ Saved features: {DATA_PROCESSED / 'training_features.csv'}")

# Quick correlation analysis
corr_matrix = features_df[['oni_jjas_avg', 'oni_lag_3', 'pre_season_rainfall', 'prev_year_onset_doy', 'onset_doy']].corr()
print("\nFeature correlations with onset_doy:")
print(corr_matrix['onset_doy'].sort_values(ascending=False))

## Random Forest Training with Leave-One-Out CV

**Why LOO-CV?** Dengan n=30, train/test split traditional (80/20) cuma kasih 6 test points, terlalu kecil untuk reliable MAE. LOO-CV pakai 29 train + 1 test, repeat 30 kali → unbiased estimate.

In [ ]:
feature_cols = ['oni_jjas_avg', 'oni_lag_3', 'pre_season_rainfall', 'prev_year_onset_doy']
X = features_df[feature_cols].values
y = features_df['onset_doy'].values
years = features_df['year'].values

# Random Forest configuration
rf_params = {
    'n_estimators': 100,
    'max_depth': 5,
    'min_samples_split': 3,
    'min_samples_leaf': 2,
    'random_state': 42
}

# LOO-CV
loo = LeaveOneOut()
predictions = []

for train_idx, test_idx in loo.split(X):
    rf = RandomForestRegressor(**rf_params)
    rf.fit(X[train_idx], y[train_idx])
    pred = rf.predict(X[test_idx])[0]
    actual = y[test_idx][0]
    test_year = years[test_idx][0]
    predictions.append({
        'year': test_year,
        'actual_doy': actual,
        'predicted_doy': pred,
        'error_days': pred - actual,
        'abs_error_days': abs(pred - actual)
    })

loo_results = pd.DataFrame(predictions)
mae = loo_results['abs_error_days'].mean()
rmse = np.sqrt((loo_results['error_days'] ** 2).mean())

print(f"📊 LOO-CV Results (n={len(loo_results)}):")
print(f"   MAE: {mae:.2f} days")
print(f"   RMSE: {rmse:.2f} days")
print(f"   Best prediction: {loo_results['abs_error_days'].min():.1f} days off")
print(f"   Worst prediction: {loo_results['abs_error_days'].max():.1f} days off")

In [ ]:
# Baseline comparison: Pranata Mangsa traditional (always predict DOY 295)
TRADITIONAL_DOY = 295
loo_results['traditional_pred'] = TRADITIONAL_DOY
loo_results['traditional_abs_error'] = abs(loo_results['actual_doy'] - TRADITIONAL_DOY)
traditional_mae = loo_results['traditional_abs_error'].mean()

print(f"\n📊 Baseline Comparison:")
print(f"   Pranata Mangsa traditional MAE: {traditional_mae:.2f} days")
print(f"   Mangsakala RF MAE: {mae:.2f} days")
print(f"   Improvement: {traditional_mae - mae:.2f} days ({(traditional_mae - mae)/traditional_mae*100:.1f}%)")

# Linear regression baseline (sanity check — ML harus beat linear)
lr_predictions = []
for train_idx, test_idx in loo.split(X):
    lr = LinearRegression()
    lr.fit(X[train_idx], y[train_idx])
    pred = lr.predict(X[test_idx])[0]
    actual = y[test_idx][0]
    lr_predictions.append(abs(pred - actual))

lr_mae = np.mean(lr_predictions)
print(f"   Linear Regression MAE: {lr_mae:.2f} days")
if mae < lr_mae:
    print(f"   ✅ RF beats linear regression (good — non-linear pattern captured)")
else:
    print(f"   ⚠️ RF doesn't beat linear regression (consider simpler model)")

In [ ]:
# Visualize predictions vs actuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time series
ax = axes[0]
ax.plot(loo_results['year'], loo_results['actual_doy'], 'o-', label='Actual', color='#7A8450')
ax.plot(loo_results['year'], loo_results['predicted_doy'], 's--', label='Predicted (LOO-CV)', color='#C8553D')
ax.axhline(295, color='gray', linestyle=':', label='Traditional (DOY 295)')
ax.set_xlabel('Year')
ax.set_ylabel('Onset DOY')
ax.set_title(f'Predicted vs Actual Onset (LOO-CV, MAE={mae:.1f} days)')
ax.legend()
ax.grid(alpha=0.3)

# Scatter
ax = axes[1]
ax.scatter(loo_results['actual_doy'], loo_results['predicted_doy'], color='#5B8AA6', s=60, edgecolor='black')
lims = [loo_results['actual_doy'].min() - 10, loo_results['actual_doy'].max() + 10]
ax.plot(lims, lims, 'k--', alpha=0.5, label='Perfect prediction')
ax.set_xlabel('Actual Onset DOY')
ax.set_ylabel('Predicted Onset DOY')
ax.set_title('Predicted vs Actual')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/loo_cv_results.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Save LOO results
loo_results.to_csv(DATA_PROCESSED / 'loo_cv_results.csv', index=False)
print(f"✅ Saved LOO results: {DATA_PROCESSED / 'loo_cv_results.csv'}")

## Train Final Model on All Data

In [ ]:
# Train on full dataset for production
final_rf = RandomForestRegressor(**rf_params)
final_rf.fit(X, y)

# Feature importance
importances = pd.Series(final_rf.feature_importances_, index=feature_cols).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot(kind='barh', ax=ax, color='#7A8450')
ax.set_title('Feature Importance — Random Forest')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('../docs/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nFeature importance:")
print(importances.sort_values(ascending=False))

In [ ]:
# Save model
joblib.dump(final_rf, MODEL_DIR / 'onset_predictor.pkl')
print(f"✅ Saved model: {MODEL_DIR / 'onset_predictor.pkl'}")

# Save feature columns (for inference)
with open(MODEL_DIR / 'feature_columns.json', 'w') as f:
    json.dump(feature_cols, f)
print(f"✅ Saved feature columns: {MODEL_DIR / 'feature_columns.json'}")

# Save metadata
metadata = {
    'model_version': 'rf-v1.0',
    'training_date': datetime.now().isoformat(),
    'n_training_samples': len(features_df),
    'features': feature_cols,
    'target': 'onset_doy',
    'rf_params': rf_params,
    'metrics': {
        'loo_cv_mae_days': float(mae),
        'loo_cv_rmse_days': float(rmse),
        'baseline_traditional_mae_days': float(traditional_mae),
        'baseline_linear_mae_days': float(lr_mae),
        'improvement_vs_traditional_days': float(traditional_mae - mae)
    },
    'limitations': [
        'Limited training data (n=30 onset events)',
        'Single region validation (Kebumen only)',
        'CHIRPS resolution 5.5km may miss micro-climate',
        'Onset detection rule simplified vs full Liebmann-Marengo',
        'Does not incorporate IOD (Indian Ocean Dipole)'
    ]
}

with open(MODEL_DIR / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✅ Saved metadata: {MODEL_DIR / 'model_metadata.json'}")

## Inference Test

Verify model bisa di-load dan predict untuk tahun future (2026).

In [ ]:
# Test inference
loaded_model = joblib.load(MODEL_DIR / 'onset_predictor.pkl')

# Build feature for 2026 (use latest available ONI data, or extrapolate)
# For now, using mock 2026 features
test_features_2026 = {
    'oni_jjas_avg': 0.5,         # weak El Niño expected
    'oni_lag_3': 0.3,
    'pre_season_rainfall': 120,  # mm
    'prev_year_onset_doy': features_df['onset_doy'].iloc[-1]  # 2025 onset
}

X_2026 = np.array([[test_features_2026[col] for col in feature_cols]])
pred_2026 = loaded_model.predict(X_2026)[0]
pred_date_2026 = pd.Timestamp(f'2026-01-01') + pd.Timedelta(days=pred_2026 - 1)

print(f"📅 Predicted onset for 2026 (test scenario):")
print(f"   Features: {test_features_2026}")
print(f"   Predicted DOY: {pred_2026:.1f}")
print(f"   Predicted date: {pred_date_2026.date()}")
print(f"   Confidence interval: ±{mae:.1f} days (LOO-CV MAE)")
print(f"   Shift from traditional: {pred_2026 - 295:+.1f} days")

## Summary Report

Section ini buat lo paste ke README atau video script.

### Model Performance Honest Disclosure

Berdasarkan output cell di atas:
- **Training data:** n=30 onset events (1995-2025)
- **MAE LOO-CV:** lihat output `mae` variable di cell sebelumnya
- **Baseline (Pranata Mangsa):** lihat output `traditional_mae`
- **Improvement vs baseline:** lihat output `(traditional_mae - mae)/traditional_mae`

### Honest Statements untuk Video & README

**Yang OK diklaim:**
- ✅ "Mangsakala MAE [X] days, vs Pranata Mangsa traditional MAE [Y] days"
- ✅ "Validated using Leave-One-Out CV on 30 onset events"
- ✅ "Model captures ENSO-driven onset variability (feature importance shows ONI features)"

**Yang HARUS disclose:**
- ⚠️ "Limited training data (n=30) — production deployment requires extended validation"
- ⚠️ "Single region (Kebumen) — generalization to other regions not yet tested"
- ⚠️ "Onset detection rule simplified vs full Liebmann-Marengo"

**Yang JANGAN diklaim:**
- ❌ "Model akurat 95%" (tidak ada justifikasi untuk angka ini)
- ❌ "Production-ready" (n=30 terlalu kecil untuk production)
- ❌ "Outperform semua kalender tradisional" (cuma tested di Pranata Mangsa Kebumen)

**Next step:** Reza integrate `model/onset_predictor.pkl` ke FastAPI backend (lihat EXECUTION_GUIDE Section 4.2).